# 08. Rate Limiter

Provider APIs enforce hard ceilings on requests per minute. Without a
client-side throttle, a fleet of agents will blast the API, get a wall
of `429 Too Many Requests` back, and waste real money on retry storms.
`RateLimitModule` solves this with the classic **token bucket** algorithm:
fast when capacity is available, well-behaved when it isn't.

**You will learn:**
- How a token bucket allows bursts while enforcing a sustained average rate
- How `RateLimitModule` wires a bucket per provider and gates every `invoke()`
- The two config knobs (`requests_per_minute`, `burst_capacity`) and their
  validation rules
- Why two adapters for the same provider share one bucket, and how isolation
  works across providers
- Why rate limiting must sit **innermost** in the module stack, just above
  the adapter

Every cell runs deterministically without an API key. We patch
`time.monotonic` and `asyncio.sleep` so the bucket's clock is under our
control.

## 1. Setup

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

We will rely on three knobs throughout this notebook:

- `unittest.mock.patch` to swap in a deterministic clock for `time.monotonic`
- `AsyncMock` adapters that satisfy the `LLMProvider` protocol without making
  HTTP calls
- `arcllm.modules.rate_limit.clear_buckets()` to reset the per-provider
  shared registry between scenarios

Import the public API once so later cells stay focused on behaviour:

In [ ]:
from unittest.mock import AsyncMock, MagicMock, patch

from arcllm.exceptions import ArcLLMConfigError
from arcllm.modules.rate_limit import (
    RateLimitModule,
    TokenBucket,
    clear_buckets,
)
from arcllm.types import LLMProvider, LLMResponse, Message, Usage

clear_buckets()
print("rate_limit module imports ready")

## 2. Token bucket primer

The token bucket has been the rate-limiting workhorse for decades. It powers
Linux `tc`, nginx `limit_req`, AWS API Gateway, and most cloud SDK
throttles. The mental model:

```
capacity=5, refill_rate=1 token/sec

t=0.0  [#####]  5/5   bucket starts full
t=0.0  [####.]  4/5   request consumes one token
t=0.0  [###..]  3/5   burst: another request, no wait
t=0.0  [##...]  2/5
t=0.0  [#....]  1/5
t=0.0  [.....]  0/5   bucket empty
t=0.0  wait...        next request blocks
t=1.0  [#....]  1/5   refill produced one token
t=1.0  [.....]  0/5   request consumed it
```

Two knobs control behaviour:

- **`capacity`** -- the maximum bucket size. This is the **burst allowance**:
  the most requests that can fire instantly when the bucket is full.
- **`refill_rate`** -- tokens added per second. This is the **sustained
  average rate**. For a 60 RPM target, `refill_rate = 60 / 60 = 1.0`.

Compared to the alternatives:

| Algorithm | Burst | Memory | Edge cases |
|---|---|---|---|
| Fixed window | up to 2x at boundary | counter + window start | Boundary spikes |
| Sliding window log | none | every request timestamp | Memory grows with QPS |
| Leaky bucket | none | counter + last drain | No burst smoothing |
| **Token bucket** | up to `capacity` | counter + last refill | None — clean math |


Construct one directly. The constructor stamps `time.monotonic()` as
`_last_refill` and pre-fills the bucket to `capacity`.

In [ ]:
bucket = TokenBucket(capacity=5, refill_rate=1.0)
print(f"capacity   : {bucket._capacity}")
print(f"tokens     : {bucket._tokens}")
print(f"refill rate: {bucket._refill_rate} tokens/sec")
print(f"lock type  : {type(bucket._lock).__name__}")

`time.monotonic()` is the right clock for rate limiting. Unlike
`time.time()`, it is immune to wall-clock adjustments (NTP corrections,
manual clock changes). Two consecutive `monotonic()` calls always return
non-decreasing values.

## 3. RateLimitModule config

`RateLimitModule` is a thin wrapper around `TokenBucket`. It implements
the `LLMProvider` protocol and, for every `invoke()`, acquires one token
before delegating to the inner adapter.

Two config keys:

| Key | Required | Default | Constraint |
|---|---|---|---|
| `requests_per_minute` | yes | `60` | `> 0` |
| `burst_capacity` | no | `requests_per_minute` | `>= 1` |

If you only set `requests_per_minute`, the bucket capacity defaults to it.
That gives a sensible "one minute's worth of burst" budget.

Build a mock adapter that satisfies the `LLMProvider` protocol and feed it
into a module:

In [ ]:
def make_inner(name: str = "anthropic") -> MagicMock:
    """Return a MagicMock pretending to be an LLMProvider for *name*."""
    inner = MagicMock(spec=LLMProvider)
    inner.name = name
    inner.model_name = "claude-sonnet-4-6"
    inner.invoke = AsyncMock(
        return_value=LLMResponse(
            content="ok",
            usage=Usage(input_tokens=10, output_tokens=5, total_tokens=15),
            model="claude-sonnet-4-6",
            stop_reason="end_turn",
        )
    )
    return inner


clear_buckets()
inner = make_inner()
module = RateLimitModule({"requests_per_minute": 60}, inner)
print(f"name     : {module.name}")
print(f"model    : {module.model_name}")
print(f"capacity : {module._bucket._capacity}")
print(f"rate     : {module._bucket._refill_rate} tokens/sec")

Because `requests_per_minute=60`, the module configures the bucket with
capacity 60 and refill 1 token/second. That is the default Arc setting and
matches the most conservative provider tier (Anthropic free, OpenAI
free-tier, etc.).

Now exercise the validation rules. Bad inputs raise `ArcLLMConfigError`
at construction — fail-fast rather than at the first call.

In [ ]:
clear_buckets()
inner = make_inner()

for bad_config in [
    {"requests_per_minute": 0},
    {"requests_per_minute": -10},
    {"requests_per_minute": 60, "burst_capacity": 0},
]:
    try:
        RateLimitModule(bad_config, inner)
    except ArcLLMConfigError as exc:
        print(f"rejected {bad_config!r:<60s}  -> {exc}")
    else:
        print(f"WRONG: {bad_config!r} should have raised")

All three rejected at module construction time. There is no code path
where an agent can call `invoke()` against a misconfigured rate limiter.

## 4. Burst behavior

Burst is the bucket's filled-state head-start. With `burst_capacity=5`,
the first five requests acquire instantly — `acquire()` returns `0.0`
for each one. The sixth request finds an empty bucket and has to wait.

We will patch `asyncio.sleep` to a no-op so the test runs deterministically;
the wait *value* still tells us what would have happened in production.

In [ ]:
async def drain_immediately(bucket: TokenBucket, n: int) -> list[float]:
    """Acquire n tokens and return the wait-time observed for each."""
    waits: list[float] = []
    for _ in range(n):
        waits.append(await bucket.acquire())
    return waits


clear_buckets()
bucket = TokenBucket(capacity=5, refill_rate=1.0)
burst_waits = await drain_immediately(bucket, 5)
print(f"burst waits: {burst_waits}")
print(f"tokens left after burst: {bucket._tokens:.4f}")

All five returns are `0.0` — the bucket served them from prefill. The
tokens-remaining count is a tiny epsilon above zero because a tiny amount
of wall-clock time advanced inside `_refill()`.

Now ask for a sixth token with the clock advanced by exactly one second.
We patch `time.monotonic` and `asyncio.sleep` together: `_refill()` sees
the new time, but `acquire()` doesn't actually pause.

In [ ]:
# Build a fresh bucket entirely under a controlled clock so every
# call to time.monotonic() — including the constructor's — comes from
# our scripted timeline. asyncio.sleep is replaced with a hook that
# advances the clock by the requested wait, then returns immediately.
clear_buckets()

scripted_clock = {"t": 1000.0}


async def fake_sleep(seconds: float) -> None:
    scripted_clock["t"] += seconds


def now() -> float:
    return scripted_clock["t"]


with patch("arcllm.modules.rate_limit.time.monotonic", side_effect=now):
    with patch("arcllm.modules.rate_limit.asyncio.sleep", new=fake_sleep):
        bucket2 = TokenBucket(capacity=5, refill_rate=1.0)
        for _ in range(5):
            await bucket2.acquire()
        # Bucket is empty. The 6th acquire must wait — the fake sleep
        # advances the clock by the deficit-derived wait time, the loop
        # re-checks, finds one token, and returns the cumulative wait.
        wait = await bucket2.acquire()

print(f"wait observed for 6th token: {wait:.3f}s")

The caller would have slept ~1 second waiting for one token to refill at
the rate of 1 token/sec. In production this is a transparent delay — the
agent code never sees an error.

Burst is a knob worth tuning. A few common shapes:

| Profile | RPM | Burst | When to use |
|---|---|---|---|
| Default | 60 | 60 | Most providers, balanced behaviour |
| Conservative | 60 | 5 | Tight quotas, no headroom for spikes |
| Aggressive | 60 | 600 | Long idle periods + occasional bursts |

Setting `burst_capacity > requests_per_minute` only helps if the bucket is
actually allowed to fill up — i.e., the agent is mostly idle. Otherwise
the long-run rate is bounded by `refill_rate`.

## 5. Cooldown / refill behavior

Refill is what enforces the *sustained* rate. The math is:

```python
elapsed = now - last_refill
tokens  = min(capacity, tokens + elapsed * refill_rate)
```

Two important properties:

1. **Continuous refill** — there's no "window" boundary. Every call to
   `_refill()` adds whatever tokens have accrued since the last refill.
2. **Capacity cap** — idle time can't accumulate unlimited credit.
   Idle for an hour at 60 RPM with capacity=60 gives you 60 tokens, not
   3,600.

Demonstrate both with a deterministic clock:

In [ ]:
clear_buckets()

with patch("arcllm.modules.rate_limit.time.monotonic") as mock_mono:
    mock_mono.return_value = 100.0
    bucket = TokenBucket(capacity=10, refill_rate=2.0)  # 2 tokens/sec

    # Drain the bucket synchronously by mutating the float counter.
    # (acquire() is async; we want to test refill in isolation.)
    bucket._tokens = 0.0
    print(f"after drain : {bucket._tokens:.2f} tokens")

    # Advance 3 seconds -> +6 tokens
    mock_mono.return_value = 103.0
    bucket._refill()
    print(f"+3s         : {bucket._tokens:.2f} tokens (expected 6.00)")

    # Advance another 5 seconds -> +10 tokens, capped at capacity=10
    mock_mono.return_value = 108.0
    bucket._refill()
    print(f"+8s total   : {bucket._tokens:.2f} tokens (capped at 10.00)")

    # Idle for an hour. Still capped at 10 — no unbounded credit.
    mock_mono.return_value = 108.0 + 3600
    bucket._refill()
    print(f"+1h idle    : {bucket._tokens:.2f} tokens (still capped)")

The capacity cap is the design choice that makes the bucket
well-behaved over long horizons. Without it, an idle agent could store up
weeks of credit and then DDoS the provider on its first wake-up call.

**Exact wait time on an empty bucket.** If `_tokens < 1.0`, `acquire()`
computes the wait as `(1.0 - tokens) / refill_rate`:

In [ ]:
# Reuse the same scripted-clock + fake-sleep pattern. The fake sleep
# advances the clock by exactly the wait the bucket asks for, so the
# next loop iteration sees enough tokens to consume.
scripted = {"t": 0.0}


async def adv(seconds: float) -> None:
    scripted["t"] += seconds


with patch(
    "arcllm.modules.rate_limit.time.monotonic",
    side_effect=lambda: scripted["t"],
):
    with patch("arcllm.modules.rate_limit.asyncio.sleep", new=adv):
        bucket3 = TokenBucket(capacity=10, refill_rate=4.0)  # 4 tokens/sec
        bucket3._tokens = 0.25  # quarter of a token sitting there
        # Need 1 - 0.25 = 0.75 tokens. At 4/sec that's 0.1875s of wait.
        wait = await bucket3.acquire()

print(f"computed wait: {wait:.4f}s (expected ~0.1875s)")

## 6. Multi-provider isolation

Provider rate limits are enforced **per API key**, not per model. If you
have one Anthropic key and ten agents using `claude-sonnet-4-6`, they all
compete for the same 1,000 RPM. They must share one bucket.

But agents pointing at different providers (Anthropic, OpenAI, …) hit
totally separate quotas — they must **not** share a bucket.

Arc resolves this with a process-global registry keyed by provider name:

```python
_bucket_registry: dict[str, TokenBucket] = {}

def _get_or_create_bucket(provider, capacity, refill_rate):
    if provider not in _bucket_registry:
        _bucket_registry[provider] = TokenBucket(capacity, refill_rate)
    return _bucket_registry[provider]
```

Two modules with the same `provider` get the **same** bucket object. Two
modules with different providers get distinct buckets.

In [ ]:
clear_buckets()

config = {"requests_per_minute": 60}
anth_a = RateLimitModule(config, make_inner("anthropic"))
anth_b = RateLimitModule(config, make_inner("anthropic"))
openai = RateLimitModule(config, make_inner("openai"))

print(f"anth_a bucket id  : {id(anth_a._bucket)}")
print(f"anth_b bucket id  : {id(anth_b._bucket)}")
print(f"openai bucket id  : {id(openai._bucket)}")
print()
print(f"anth_a is anth_b  : {anth_a._bucket is anth_b._bucket}")
print(f"anth_a is openai  : {anth_a._bucket is openai._bucket}")

Same provider name, same bucket. Different name, different bucket.

Note the side effect: the *first* `RateLimitModule` for a provider wins
the configuration race. If `anth_a` is built with capacity 60 and
`anth_b` is later built with capacity 600, both still share the
60-capacity bucket. This is intentional — the rate limit is a
process-wide invariant, not a per-module setting.

In [ ]:
clear_buckets()

first = RateLimitModule({"requests_per_minute": 60}, make_inner("anthropic"))
second = RateLimitModule({"requests_per_minute": 600}, make_inner("anthropic"))

print(f"first  bucket capacity: {first._bucket._capacity}")
print(f"second bucket capacity: {second._bucket._capacity}")
print(f"shared object         : {first._bucket is second._bucket}")

# To pick up the new config, callers must clear and rebuild.
clear_buckets()
rebuilt = RateLimitModule({"requests_per_minute": 600}, make_inner("anthropic"))
print(f"rebuilt capacity      : {rebuilt._bucket._capacity}")

`clear_buckets()` is the official escape hatch. It is wired into
`arcllm.registry.clear_cache()` and is called by the test suite between
scenarios.

## 7. Position in module stack — why innermost

ArcLLM's module stack, from outermost to innermost:

```
Otel -> Queue -> Telemetry -> CircuitBreaker -> Audit -> Security ->
Retry -> Fallback -> RateLimit -> Adapter
```

`RateLimitModule` sits one level above the adapter. That position is
deliberate. Three reasons:

1. **Throttle before the call.** The bucket gates the actual HTTP request,
   not some upstream wrapper. We don't burn a token on a request that
   never goes out.
2. **Don't trip Fallback on a wait.** If RateLimit were *outside* Fallback,
   then a long wait would look like a stalled adapter, and Fallback would
   try a different provider. That defeats the purpose — we want the
   call to succeed against the rate-limited provider, not abandon it.
3. **Don't cause Retry storms.** If RateLimit were outside Retry, an
   actual rate-limit response from the API would skip the bucket entirely,
   then Retry would pile *more* requests onto an already-overloaded
   provider.

Sketch the wiring with a fresh stack:

In [ ]:
from arcllm.modules.fallback import FallbackModule
from arcllm.modules.retry import RetryModule

clear_buckets()

adapter = make_inner("anthropic")
rate = RateLimitModule({"requests_per_minute": 60}, adapter)
fallback = FallbackModule({"providers": ["openai"]}, rate)
retry = RetryModule({"max_attempts": 3}, fallback)

# Walk the wrapping order from outermost to innermost.
current = retry
depth = 0
while True:
    print(f"{'  ' * depth}{type(current).__name__}")
    inner = getattr(current, "_inner", None)
    if inner is None or inner is current:
        break
    current = inner
    depth += 1

Reading the indented output: `Retry` wraps `Fallback`, which wraps
`RateLimit`, which wraps the adapter (the `MagicMock`). Every `invoke()`
passes through `Retry -> Fallback -> RateLimit -> adapter` in that order,
and back out the same way for return values.

**What if RateLimit moved outside Fallback?** Imagine `Fallback ->
RateLimit -> Retry -> adapter`. The bucket gates traffic only after
Fallback has already chosen a provider, but Fallback's whole job is to
*switch* providers when one is misbehaving. The bucket would be configured
for one provider but tokens would be spent on calls that ended up at
another. The accounting would silently lie.

**`registry.load_model` honours this order automatically.** When you set
`rate_limit=True` on `load_model()`, the rate limiter is wrapped innermost
regardless of which other modules you enable:

In [ ]:
import os
from arcllm.registry import clear_cache, load_model

os.environ.setdefault("ANTHROPIC_API_KEY", "test-not-real")
clear_cache()
clear_buckets()

model = load_model(
    "anthropic",
    retry=True,
    fallback=False,
    rate_limit={"requests_per_minute": 120, "burst_capacity": 30},
)

current = model
depth = 0
while current is not None:
    print(f"{'  ' * depth}{type(current).__name__}")
    next_inner = getattr(current, "_inner", None)
    if next_inner is None or next_inner is current:
        break
    current = next_inner
    depth += 1

The adapter (`AnthropicAdapter`) is innermost; `RateLimitModule` is the
next layer up; `RetryModule` sits outside both.

## 8. End-to-end — a throttled `invoke()`

Tie everything together. Configure a `burst_capacity=1` module so the
second call is guaranteed to wait. We patch the clock and `sleep` so the
test runs in milliseconds, but the WARN log and span attribute carry the
real-world wait time.

In [ ]:
import logging
import io

clear_buckets()

buf = io.StringIO()
handler = logging.StreamHandler(buf)
handler.setLevel(logging.WARNING)
rl_logger = logging.getLogger("arcllm.modules.rate_limit")
rl_logger.addHandler(handler)
rl_logger.setLevel(logging.WARNING)

inner = make_inner("anthropic")

clock = {"t": 0.0}


async def advance(seconds: float) -> None:
    clock["t"] += seconds


msg = [Message(role="user", content="hi")]
with patch(
    "arcllm.modules.rate_limit.time.monotonic",
    side_effect=lambda: clock["t"],
):
    with patch("arcllm.modules.rate_limit.asyncio.sleep", new=advance):
        # Construct under the patched clock so the bucket starts at t=0.
        module = RateLimitModule(
            {"requests_per_minute": 60, "burst_capacity": 1},
            inner,
        )
        await module.invoke(msg)  # first: free (uses prefill token)
        await module.invoke(msg)  # second: must wait ~1s for refill

rl_logger.removeHandler(handler)
print("captured warning(s):")
print(buf.getvalue() or "<none>")
print(f"simulated clock now: {clock['t']:.3f}s")
print(f"inner.invoke call count: {inner.invoke.call_count}")

The WARNING line carries enough context for an operator to diagnose
throttling without enabling DEBUG logs:

```
Rate limited for provider 'anthropic'. Waited 1.00s for token.
```

The same number is also recorded as `arcllm.rate_limit.wait_ms` on the
OTel span (see notebook 11 for OTel export). That makes it easy to chart
throttle pressure across providers.

## 9. Concurrent acquisition

`TokenBucket` uses `asyncio.Lock` so multiple coroutines sharing one
bucket don't corrupt the counter. The locking discipline is precise:

```python
while True:
    async with self._lock:
        self._refill()
        if self._tokens >= 1.0:
            self._tokens -= 1.0
            return total_wait
        wait_seconds = (1.0 - self._tokens) / self._refill_rate
    # sleep OUTSIDE the lock so other waiters can re-check
    await asyncio.sleep(wait_seconds)
    total_wait += wait_seconds
```

Sleeping outside the lock is the difference between "works under load"
and "deadlocks under load". If the lock were held during sleep, every
waiter would wake one-at-a-time, in lock-acquisition order, instead of
competing fairly for newly-refilled tokens.

Five coroutines racing for one starting token, with a deterministic clock:

In [ ]:
import asyncio

clear_buckets()

race_clock = {"t": 0.0}


async def race_sleep(seconds: float) -> None:
    # The clock only advances when the bucket actually sleeps.
    race_clock["t"] += seconds


with patch(
    "arcllm.modules.rate_limit.time.monotonic",
    side_effect=lambda: race_clock["t"],
):
    with patch("arcllm.modules.rate_limit.asyncio.sleep", new=race_sleep):
        # capacity=1, refill_rate=2/sec: one token at t=0, then 1 more
        # token every 0.5s.
        bucket = TokenBucket(capacity=1, refill_rate=2.0)
        results = await asyncio.gather(*(bucket.acquire() for _ in range(5)))

for i, wait in enumerate(results):
    print(f"coro {i}: wait={wait:.4f}s")
print(f"simulated clock end: {race_clock['t']:.4f}s")

The first coroutine to enter the lock consumes the prefilled token (no
wait); the rest queue up and observe progressively larger waits as the
clock advances. No coroutine sees a corrupted token count, and nobody
deadlocks.

## 10. Summary

**What `RateLimitModule` is:** a token-bucket throttle that gates every
`invoke()` call before the request reaches the adapter. Two config keys,
one shared bucket per provider, one `clear_buckets()` escape hatch for
tests and registry resets.

**What you learned in this notebook:**

- A token bucket allows bursts up to `capacity` and enforces a sustained
  average rate of `refill_rate` tokens/sec.
- `RateLimitModule` validates `requests_per_minute > 0` and
  `burst_capacity >= 1` at construction, raising `ArcLLMConfigError`.
- Burst lets a workload absorb spikes; the long-run average is bounded by
  `refill_rate` regardless.
- Refill is continuous and capped at `capacity` — idle time can't earn
  unbounded credit.
- The bucket registry keys on provider name. Two modules for the same
  provider share state; different providers stay isolated.
- The first module per provider wins the bucket configuration. Use
  `clear_buckets()` (or `clear_cache()`) to rebuild with new settings.
- `RateLimitModule` sits **innermost** in the stack — wrapped by Retry,
  Fallback, Security, Audit, and the rest — so it throttles the actual
  outbound call without distorting upper-layer accounting.
- Throttling emits a `WARNING` log and an OTel span event
  (`arcllm.rate_limit.throttled`) with the wait duration.
- The `asyncio.Lock` is held only during the refill+check; `asyncio.sleep`
  happens outside the lock so concurrent waiters compete fairly.

**API surface covered:**

- `arcllm.modules.rate_limit.RateLimitModule`
- `arcllm.modules.rate_limit.TokenBucket`
- `arcllm.modules.rate_limit.clear_buckets`
- Config keys: `requests_per_minute`, `burst_capacity`
- `arcllm.exceptions.ArcLLMConfigError` (raised on invalid config)
- `arcllm.registry.load_model(rate_limit=...)` integration

**Next:** notebook 09 covers `TelemetryModule`, the next layer out — where
the per-call timings (including any rate-limit wait) get aggregated for
cost and latency reporting.